# Kaggle Public Leaderboard Scores

| Task                               | Submission File          | Score |
| ---------------------------------- | ------------------------ | ----: |
| Task 1                             | `LogReg_Prediction.csv`  | 0.620 |
| Task 2 (Feature Selection, 2000)   | `task2_knn_fs_2000.csv`  | 0.549 |
| Task 2 (Feature Selection, 1000)   | `task2_knn_fs_1000.csv`  | 0.496 |
| Task 2 (Feature Selection, 500)    | `task2_knn_fs_500.csv`   | 0.436 |
| Task 2 (Feature Selection, 100)    | `task2_knn_fs_100.csv`   | 0.238 |
| Task 2 (Dimension Reduction, 2000) | `task2_knn_svd_2000.csv` | 0.581 |
| Task 2 (Dimension Reduction, 1000) | `task2_knn_svd_1000.csv` | 0.567 |
| Task 2 (Dimension Reduction, 500)  | `task2_knn_svd_500.csv`  | 0.562 |
| Task 2 (Dimension Reduction, 100)  | `task2_knn_svd_100.csv`  | 0.522 |
| Task 3                             | `task3_linear_svc.csv`   | 0.677 |
| Task 3                             | `task3_complement_nb.csv`   | 0.560 |
| Task 3                             | `task3_random_forest.csv`   | 0.490 |
| Task 3                             | `task3_xgboost_final.csv`   | 0.547 |

# Setup


In [4]:
from pathlib import Path
import ast
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import (
    HalvingGridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
    cross_val_score
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, normalize
from sklearn.svm import LinearSVC

In [5]:
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
OUTPUTS_TASK2_DIR = PROJECT_ROOT / "outputs" / "task2"
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"

TFIDF_DEFAULTS = {
    "max_features": 5000,
    "min_df": 2,
    "sublinear_tf": True,
}

RANDOM_STATE = 42  # for reproducibility. set to None for random

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

# Preprocessing


In [6]:
def _clean_text(text: object) -> str:
    if not isinstance(text, str):
        return ""

    t = text.lower()

    # Remove LaTeX math blocks and commands.
    t = re.sub(r"\$.*?\$|\\\[.*?\\\]|\\\(.*?\\\)", " ", t, flags=re.DOTALL)
    t = re.sub(
        r"\\(?:begin|end)\{[^{}]*\}|\\[a-zA-Z]+\*?(?:\[[^\]]*\])?(?:\{[^{}]*\})*",
        " ",
        t,
    )

    # Remove URLs and numeric citation brackets.
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"\[[0-9,\s]+\]", " ", t)

    # Keep only alphanumeric, hyphen, underscore, and spaces.
    t = re.sub(r"[^a-z0-9\-_ ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def load_train_test_cleaned_data_local(
    project_root: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_cache = project_root / "data" / "cache" / "train_cleaned.csv"
    test_cache = project_root / "data" / "cache" / "test_cleaned.csv"

    if train_cache.exists() and test_cache.exists():
        train_df = pd.read_csv(train_cache, sep="	")
        test_df = pd.read_csv(test_cache, sep="	")
        train_df = train_df[["id", "label_id", "cleaned_abstract"]].copy()
        test_df = test_df[["id", "cleaned_abstract"]].copy()
        return train_df, test_df

    train_path = project_root / "data" / "train.csv"
    test_path = project_root / "data" / "test.csv"
    train_raw = pd.read_csv(train_path, sep="\t")
    test_raw = pd.read_csv(test_path, sep="\t")

    train_df = train_raw[["id", "label_id"]].copy()
    test_df = test_raw[["id"]].copy()
    train_df["cleaned_abstract"] = train_raw["abstract"].map(_clean_text)
    test_df["cleaned_abstract"] = test_raw["abstract"].map(_clean_text)
    return train_df, test_df


train_df, test_df = load_train_test_cleaned_data_local(PROJECT_ROOT)
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Classes: {train_df['label_id'].nunique()}")

Train shape: (139156, 3)
Test shape: (34790, 2)
Classes: 39


# Task 1: Custom Logistic Regression


## Task 1 Hyperparameters


In [28]:
EPOCHS = 10
BATCH_SIZE = 512
LEARNING_RATE = 2.0
REG_STRENGTH = 1e-4

## Binary Logistic Regression


In [29]:
class LogisticRegression:
    def __init__(self):
        self.weights = None
        self.bias = None

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def gradients(self, X, y, y_hat, reg_strength, sample_weight):
        weight_sum = float(np.sum(sample_weight))
        error = (y_hat - y) * sample_weight
        dw = (1.0 / weight_sum) * (X.T @ error) + reg_strength * self.weights
        db = (1.0 / weight_sum) * np.sum(error)
        return dw, db

    def train(self, X, y, bs, epochs, lr, reg_strength=1e-4):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=np.float64)
        self.bias = 0.0

        y_arr = np.asarray(y)
        pos_count = int(np.sum(y_arr == 1))
        neg_count = n_samples - pos_count

        pos_w = n_samples / (2.0 * pos_count)
        neg_w = n_samples / (2.0 * neg_count)

        rng = np.random.default_rng(RANDOM_STATE)
        for _ in range(epochs):
            order = rng.permutation(n_samples)
            X_epoch = X[order]
            y_epoch = y_arr[order]

            for i in range(0, n_samples, bs):
                X_batch = X_epoch[i : i + bs]
                y_batch = y_epoch[i : i + bs]
                sample_weight = np.where(y_batch == 1, pos_w, neg_w).astype(np.float64)

                y_hat = self.sigmoid(X_batch @ self.weights + self.bias)
                dw, db = self.gradients(
                    X_batch, y_batch, y_hat, reg_strength, sample_weight
                )
                self.weights -= lr * dw
                self.bias -= lr * db

## OvR Multiclass Adaptation


In [30]:
class MultiClassLogisticRegression:
    def __init__(self):
        self.models = []
        self.n_classes = None

    def train(self, X, y, bs=32, epochs=100, lr=0.01, reg_strength=1e-4):
        unique_classes = np.unique(y)
        self.n_classes = len(unique_classes)
        self.models = []

        for c in range(self.n_classes):
            y_binary = (y == c).astype(int)
            model = LogisticRegression()
            model.train(
                X, y_binary, bs=bs, epochs=epochs, lr=lr, reg_strength=reg_strength
            )
            self.models.append(model)

    def predict(self, X):
        if self.n_classes is None:
            raise ValueError("Model is not trained yet.")

        n_samples = int(X.shape[0])
        n_classes = int(self.n_classes)
        probs = np.zeros((n_samples, n_classes), dtype=np.float64)
        for c, model in enumerate(self.models):
            linear_output = X @ model.weights + model.bias
            probs[:, c] = model.sigmoid(linear_output)
        return np.argmax(probs, axis=1)

## sklearn Logistic Regression Comparison


In [31]:
# Prepare data for task 1
X_text = train_df["cleaned_abstract"].to_numpy(dtype=str)
y = train_df["label_id"].to_numpy(dtype=np.int64)

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer_task1 = TfidfVectorizer(**TFIDF_DEFAULTS)
X_train = vectorizer_task1.fit_transform(X_train_text)
X_val = vectorizer_task1.transform(X_val_text)

In [32]:
# Predict with custom OvR logreg
custom_model = MultiClassLogisticRegression()
custom_model.train(
    X_train,
    y_train,
    bs=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    reg_strength=REG_STRENGTH,
)
custom_pred = custom_model.predict(X_val).astype(np.int64)

In [33]:
# Predict with sklearn OvR logreg for comparison
base_lr = SkLogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
)
sk_model = OneVsRestClassifier(base_lr)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_val).astype(np.int64)

In [34]:
comparison_df = pd.DataFrame(
    [
        {
            "model": "custom_logreg_ovr",
            "macro_f1": f1_score(y_val, custom_pred, average="macro"),
            "accuracy": accuracy_score(y_val, custom_pred),
        },
        {
            "model": "sklearn_logreg_ovr",
            "macro_f1": f1_score(y_val, sk_pred, average="macro"),
            "accuracy": accuracy_score(y_val, sk_pred),
        },
    ]
).sort_values(by=["macro_f1", "accuracy"], ascending=False)

comparison_df

,model,macro_f1,accuracy
1,sklearn_logreg_ovr,0.634325,0.723556
0,custom_logreg_ovr,0.616918,0.693123


## Task 1 Test Prediction Export (`LogReg_Prediction.csv`)


In [35]:
vectorizer_full = TfidfVectorizer(**TFIDF_DEFAULTS)
X_full_train = vectorizer_full.fit_transform(
    train_df["cleaned_abstract"].to_numpy(dtype=str)
)
X_full_test = vectorizer_full.transform(test_df["cleaned_abstract"].to_numpy(dtype=str))
y_full_train = train_df["label_id"].to_numpy(dtype=np.int64)

final_custom_model = MultiClassLogisticRegression()
final_custom_model.train(
    X_full_train,
    y_full_train,
    bs=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    reg_strength=REG_STRENGTH,
)
final_test_pred = final_custom_model.predict(X_full_test).astype(np.int64)

logreg_submission = pd.DataFrame(
    {
        "id": test_df["id"].astype(np.int64),
        "label_id": final_test_pred,
    }
)

if list(logreg_submission.columns) != ["id", "label_id"]:
    raise ValueError("Task 1 submission columns must be exactly: id,label_id")
if len(logreg_submission) != len(test_df):
    raise ValueError("Task 1 submission row count mismatch.")

logreg_output_path = SUBMISSIONS_DIR / "LogReg_Prediction.csv"
logreg_submission.to_csv(logreg_output_path, index=False)
print(f"Saved: {logreg_output_path}")

Saved: g:\Projects\Dev\sutd\ml\ml-project\submissions\LogReg_Prediction.csv


# Task 2: KNN with Feature Selection and Dimension Reduction


## Task 2 Hyperparameters


In [36]:
TASK2_SIZES = [2000, 1000, 500, 100]

N_SPLITS = 5

# KNN settings for Task 2
N_NEIGHBORS = 2
KNN_WEIGHTS = "distance"
KNN_METRIC = "cosine"
KNN_ALGORITHM = "brute"
KNN_N_JOBS = 4

# If false reads from output folder, else recompute results (which are saved to output folder)
RECOMPUTE_TASK2_RESULTS = False

RUN_TASK2_SUBMISSION_GENERATION = False


def build_knn(n_jobs: int = KNN_N_JOBS) -> KNeighborsClassifier:
    return KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights=KNN_WEIGHTS,
        metric=KNN_METRIC,
        algorithm=KNN_ALGORITHM,
        n_jobs=n_jobs,
    )

## Task 2 Validation Experiments (CV)


### Feature Selection (Top-N TF-IDF)


In [37]:
def run_feature_selection_cv_knn(train_df: pd.DataFrame) -> pd.DataFrame:
    texts = train_df["cleaned_abstract"].tolist()
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_indices = list(skf.split(texts, y))

    rows = []
    for max_features in TASK2_SIZES:
        fold_f1 = []
        fold_acc = []

        for train_idx, val_idx in fold_indices:
            train_texts = [texts[i] for i in train_idx]
            val_texts = [texts[i] for i in val_idx]
            y_train = y[train_idx]
            y_val = y[val_idx]

            vec = TfidfVectorizer(**{**TFIDF_DEFAULTS, "max_features": max_features})
            X_train = vec.fit_transform(train_texts)
            X_val = vec.transform(val_texts)

            clf = build_knn(n_jobs=KNN_N_JOBS)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_val).astype(np.int64)

            fold_f1.append(f1_score(y_val, y_pred, average="macro"))
            fold_acc.append(accuracy_score(y_val, y_pred))

        rows.append(
            {
                "method": "feature_selection",
                "max_features": int(max_features),
                "mean_macro_f1": float(np.mean(fold_f1)),
                "std_macro_f1": float(np.std(fold_f1)),
                "mean_accuracy": float(np.mean(fold_acc)),
                "std_accuracy": float(np.std(fold_acc)),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(by="max_features", ascending=False)
        .reset_index(drop=True)
    )

### Dimension Reduction (TruncatedSVD)


In [38]:
def _build_dim_reduction_pipeline(n_components: int) -> Pipeline:
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(**TFIDF_DEFAULTS)),
            (
                "svd",
                TruncatedSVD(
                    n_components=n_components, n_iter=7, random_state=RANDOM_STATE
                ),
            ),
            ("normalize", Normalizer(norm="l2")),
            (
                "knn",
                build_knn(n_jobs=1),
            ),
        ]
    )


def run_dimension_reduction_cv_knn(train_df: pd.DataFrame) -> pd.DataFrame:
    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"macro_f1": "f1_macro", "accuracy": "accuracy"}

    rows = []
    for n_components in TASK2_SIZES:
        pipe = _build_dim_reduction_pipeline(n_components)
        cv_result = cross_validate(
            estimator=pipe,
            X=X,
            y=y,
            cv=cv,
            scoring=scoring,
            n_jobs=KNN_N_JOBS,
            return_train_score=False,
            error_score="raise",
        )

        rows.append(
            {
                "method": "dimension_reduction",
                "n_components": int(n_components),
                "mean_macro_f1": float(np.mean(cv_result["test_macro_f1"])),
                "std_macro_f1": float(np.std(cv_result["test_macro_f1"])),
                "mean_accuracy": float(np.mean(cv_result["test_accuracy"])),
                "std_accuracy": float(np.std(cv_result["test_accuracy"])),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(by="n_components", ascending=False)
        .reset_index(drop=True)
    )

### CV Result Reporting (Load Existing CSVs)


In [39]:
fs_cv_path = OUTPUTS_TASK2_DIR / "feature_selection_knn_cv_results.csv"
svd_cv_path = OUTPUTS_TASK2_DIR / "dimension_reduction_knn_cv_results_sklearn.csv"

if RECOMPUTE_TASK2_RESULTS:
    OUTPUTS_TASK2_DIR.mkdir(parents=True, exist_ok=True)
    run_feature_selection_cv_knn(train_df).to_csv(fs_cv_path, index=False)
    run_dimension_reduction_cv_knn(train_df).to_csv(svd_cv_path, index=False)

fs_raw = pd.read_csv(fs_cv_path)
svd_raw = pd.read_csv(svd_cv_path)

fs_results = fs_raw[
    [
        "max_features",
        "mean_macro_f1",
        "std_macro_f1",
        "mean_accuracy",
        "std_accuracy",
    ]
].copy()
fs_results.insert(0, "method", "feature_selection")

svd_results = svd_raw[
    [
        "n_components",
        "mean_macro_f1",
        "std_macro_f1",
        "mean_accuracy",
        "std_accuracy",
    ]
].copy()
svd_results.insert(0, "method", "dimension_reduction")

# Combined compact report table
fs_compact = fs_results.rename(columns={"max_features": "size"}).copy()
fs_compact["method"] = "FS"
svd_compact = svd_results.rename(columns={"n_components": "size"}).copy()
svd_compact["method"] = "SVD"

combined_task2_report = pd.concat([fs_compact, svd_compact], ignore_index=True)
combined_task2_report = combined_task2_report[
    ["method", "size", "mean_macro_f1", "std_macro_f1", "mean_accuracy", "std_accuracy"]
]
combined_task2_report = combined_task2_report.sort_values(
    by=["method", "size"], ascending=[True, False]
).reset_index(drop=True)

print("Combined Task 2 report table:")
display(combined_task2_report)

Combined Task 2 report table:


,method,size,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy
0,FS,2000,0.529799,0.004757,0.620677,0.002527
1,FS,1000,0.483986,0.002018,0.581707,0.001167
2,FS,500,0.421616,0.002511,0.526991,0.001993
3,FS,100,0.229744,0.002120,0.312606,0.002710
4,SVD,2000,0.556718,0.003655,0.644593,0.002106
5,SVD,1000,0.550087,0.003417,0.641719,0.001290
6,SVD,500,0.539813,0.002111,0.637515,0.000545
7,SVD,100,0.510225,0.003399,0.618234,0.002293


## Task 2 Test Submission Generation


In [40]:
def generate_task2_feature_selection_submissions(
    train_df: pd.DataFrame, test_df: pd.DataFrame
) -> list[Path]:
    train_texts = train_df["cleaned_abstract"].tolist()
    test_texts = test_df["cleaned_abstract"].tolist()
    y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    output_paths = []
    for max_features in TASK2_SIZES:
        vec = TfidfVectorizer(**{**TFIDF_DEFAULTS, "max_features": max_features})
        X_train = vec.fit_transform(train_texts)
        X_test = vec.transform(test_texts)

        clf = build_knn(n_jobs=KNN_N_JOBS)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test).astype(np.int64)

        out_path = SUBMISSIONS_DIR / f"task2_knn_fs_{max_features}.csv"
        sub_df = pd.DataFrame(
            {"id": test_df["id"].astype(np.int64), "label_id": y_pred}
        )
        if list(sub_df.columns) != ["id", "label_id"]:
            raise ValueError("Submission columns must be exactly: id,label_id")
        if len(sub_df) != len(test_df):
            raise ValueError("Submission row count mismatch.")
        sub_df.to_csv(out_path, index=False)
        output_paths.append(out_path)

    return output_paths


def generate_task2_svd_submissions(
    train_df: pd.DataFrame, test_df: pd.DataFrame
) -> list[Path]:
    train_texts = train_df["cleaned_abstract"].tolist()
    test_texts = test_df["cleaned_abstract"].tolist()
    y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    vec = TfidfVectorizer(**TFIDF_DEFAULTS)
    X_train = vec.fit_transform(train_texts)
    X_test = vec.transform(test_texts)

    output_paths = []
    for n_components in TASK2_SIZES:
        svd = TruncatedSVD(n_components=n_components, n_iter=7)
        X_train_reduced = svd.fit_transform(X_train).astype(np.float32, copy=False)
        X_test_reduced = svd.transform(X_test).astype(np.float32, copy=False)

        normalize(X_train_reduced, norm="l2", copy=False)
        normalize(X_test_reduced, norm="l2", copy=False)

        clf = build_knn(n_jobs=KNN_N_JOBS)
        clf.fit(X_train_reduced, y_train)
        y_pred = clf.predict(X_test_reduced).astype(np.int64)

        out_path = SUBMISSIONS_DIR / f"task2_knn_svd_{n_components}.csv"
        sub_df = pd.DataFrame(
            {"id": test_df["id"].astype(np.int64), "label_id": y_pred}
        )
        if list(sub_df.columns) != ["id", "label_id"]:
            raise ValueError("Submission columns must be exactly: id,label_id")
        if len(sub_df) != len(test_df):
            raise ValueError("Submission row count mismatch.")
        sub_df.to_csv(out_path, index=False)
        output_paths.append(out_path)

    return output_paths


if RUN_TASK2_SUBMISSION_GENERATION:
    fs_paths = generate_task2_feature_selection_submissions(train_df, test_df)
    svd_paths = generate_task2_svd_submissions(train_df, test_df)
    print("Generated FS submissions:")
    for p in fs_paths:
        print(" -", p)
    print("Generated SVD submissions:")
    for p in svd_paths:
        print(" -", p)

# Task 3


## LinearSVC


### Configuration


In [41]:
LINEAR_SVC_OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "task3"
LINEAR_SVC_TUNING_OUTPUT_PATH = (
    LINEAR_SVC_OUTPUTS_DIR / "linear_svc_halving_results.csv"
)
LINEAR_SVC_VALIDATION_OUTPUT_PATH = (
    LINEAR_SVC_OUTPUTS_DIR / "linear_svc_finalists_cv5.csv"
)
LINEAR_SVC_SUBMISSION_PATH = SUBMISSIONS_DIR / "task3_linear_svc.csv"

LINEAR_SVC_N_JOBS = 2
LINEAR_SVC_SCORING = {"macro_f1": "f1_macro", "accuracy": "accuracy"}
LINEAR_SVC_TUNING_CV_SPLITS = 3
LINEAR_SVC_TUNING_FACTOR = 3
LINEAR_SVC_VALIDATION_CV_SPLITS = 5
LINEAR_SVC_VALIDATION_TOP_K = 5
LINEAR_SVC_MAX_ITER = 5000
LINEAR_SVC_TUNING_PARAM_GRID = {
    "tfidf__ngram_range": [(1, 2)],
    "tfidf__min_df": [2, 5],
    "tfidf__max_features": [75000, 100000, 150000, 200000, 300000],
    "clf__C": [0.1, 0.15, 0.2, 0.25, 0.35, 0.5, 0.75, 1.0],
}

RECOMPUTE_LINEAR_SVC_RESULTS = False
RUN_LINEAR_SVC_SUBMISSION_GENERATION = False

### Utility Functions


In [42]:
def build_linear_svc_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(**TFIDF_DEFAULTS)),
            (
                "clf",
                LinearSVC(
                    max_iter=LINEAR_SVC_MAX_ITER,
                    random_state=RANDOM_STATE,
                    class_weight="balanced",
                ),
            ),
        ]
    )


def _linear_svc_normalize_param_value(key: str, value):
    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if text == "":
            parsed = value
        elif text.lower() == "none":
            parsed = None
        else:
            try:
                parsed = ast.literal_eval(text)
            except ValueError, SyntaxError:
                parsed = value

    if key in {"tfidf__max_features", "tfidf__min_df"}:
        if parsed is None:
            return None
        if isinstance(parsed, (int, float, np.integer, np.floating)):
            parsed_float = float(parsed)
            if parsed_float.is_integer():
                return int(parsed_float)
    return parsed


def _linear_svc_extract_param_dict(row: pd.Series) -> dict:
    params = {}
    for col, value in row.items():
        if not col.startswith("param_") or pd.isna(value):
            continue
        key = col.replace("param_", "", 1)
        params[key] = _linear_svc_normalize_param_value(key, value)
    return params


def _linear_svc_top_params_from_tuning(
    tuning_df: pd.DataFrame, top_k: int
) -> list[dict]:
    sort_col = (
        "rank_test_score"
        if "rank_test_score" in tuning_df.columns
        else "mean_test_score"
    )
    ascending = sort_col == "rank_test_score"
    top_df = tuning_df.sort_values(by=sort_col, ascending=ascending).head(top_k)
    return [_linear_svc_extract_param_dict(row) for _, row in top_df.iterrows()]

### Tuning


In [43]:
def run_linear_svc_tuning(train_df: pd.DataFrame, recompute: bool) -> pd.DataFrame:
    if not recompute:
        return pd.read_csv(LINEAR_SVC_TUNING_OUTPUT_PATH)

    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    cv = StratifiedKFold(
        n_splits=LINEAR_SVC_TUNING_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    search = HalvingGridSearchCV(
        estimator=build_linear_svc_pipeline(),
        param_grid=LINEAR_SVC_TUNING_PARAM_GRID,
        cv=cv,
        scoring="f1_macro",
        n_jobs=LINEAR_SVC_N_JOBS,
        factor=LINEAR_SVC_TUNING_FACTOR,
        error_score="raise",
        refit=True,
    )
    search.fit(X, y)

    tuning_df = (
        pd.DataFrame(search.cv_results_)
        .sort_values(by=["rank_test_score", "mean_test_score"], ascending=[True, False])
        .reset_index(drop=True)
    )
    LINEAR_SVC_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    tuning_df.to_csv(LINEAR_SVC_TUNING_OUTPUT_PATH, index=False)
    return tuning_df


linear_svc_tuning_df = run_linear_svc_tuning(
    train_df,
    recompute=RECOMPUTE_LINEAR_SVC_RESULTS,
)

sort_col = (
    "rank_test_score"
    if "rank_test_score" in linear_svc_tuning_df.columns
    else "mean_test_score"
)
ascending = sort_col == "rank_test_score"
linear_svc_best_tuning = linear_svc_tuning_df.sort_values(
    by=sort_col, ascending=ascending
).head(1)

print("Task 3 best Tuning config:")
display(
    linear_svc_best_tuning[
        [
            sort_col,
            "mean_test_score",
            "param_clf__C",
            "param_tfidf__max_features",
            "param_tfidf__min_df",
            "param_tfidf__ngram_range",
        ]
    ]
)

Task 3 best Tuning config:


,rank_test_score,mean_test_score,param_clf__C,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range
0,1,0.674607,0.2,150000,2,"(1, 2)"


### Validation


In [44]:
def run_linear_svc_validation(
    train_df: pd.DataFrame, tuning_df: pd.DataFrame
) -> pd.DataFrame:
    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    candidates = _linear_svc_top_params_from_tuning(
        tuning_df, top_k=LINEAR_SVC_VALIDATION_TOP_K
    )
    cv = StratifiedKFold(
        n_splits=LINEAR_SVC_VALIDATION_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rows = []
    for idx, params in enumerate(candidates, start=1):
        pipe = build_linear_svc_pipeline()
        pipe.set_params(**params)
        cv_result = cross_validate(
            estimator=pipe,
            X=X,
            y=y,
            cv=cv,
            scoring=LINEAR_SVC_SCORING,
            n_jobs=LINEAR_SVC_N_JOBS,
            return_train_score=False,
            error_score="raise",
        )

        row = {
            "candidate_rank_tuning": idx,
            "mean_macro_f1": float(np.mean(cv_result["test_macro_f1"])),
            "std_macro_f1": float(np.std(cv_result["test_macro_f1"])),
            "mean_accuracy": float(np.mean(cv_result["test_accuracy"])),
            "std_accuracy": float(np.std(cv_result["test_accuracy"])),
            "mean_fit_time_sec": float(np.mean(cv_result["fit_time"])),
            "mean_score_time_sec": float(np.mean(cv_result["score_time"])),
        }
        for key, value in params.items():
            row[f"param_{key}"] = value
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(by=["mean_macro_f1", "mean_accuracy"], ascending=[False, False])
        .reset_index(drop=True)
    )


def select_best_linear_svc_params(
    tuning_df: pd.DataFrame,
    validation_df: pd.DataFrame,
) -> dict:
    if not validation_df.empty:
        best = validation_df.sort_values(
            by=["mean_macro_f1", "mean_accuracy"],
            ascending=[False, False],
        ).iloc[0]
        return _linear_svc_extract_param_dict(best)

    sort_col = (
        "rank_test_score"
        if "rank_test_score" in tuning_df.columns
        else "mean_test_score"
    )
    ascending = sort_col == "rank_test_score"
    best = tuning_df.sort_values(by=sort_col, ascending=ascending).iloc[0]
    return _linear_svc_extract_param_dict(best)


if RECOMPUTE_LINEAR_SVC_RESULTS:
    linear_svc_validation_df = run_linear_svc_validation(train_df, linear_svc_tuning_df)
    linear_svc_validation_df.to_csv(LINEAR_SVC_VALIDATION_OUTPUT_PATH, index=False)
else:
    linear_svc_validation_df = pd.read_csv(LINEAR_SVC_VALIDATION_OUTPUT_PATH)

linear_svc_validation_view_cols = [
    "candidate_rank_tuning",
    "mean_macro_f1",
    "std_macro_f1",
    "mean_accuracy",
    "std_accuracy",
    "param_clf__C",
    "param_tfidf__max_features",
    "param_tfidf__min_df",
    "param_tfidf__ngram_range",
]
if (
    "candidate_rank_tuning" not in linear_svc_validation_df.columns
    and "candidate_rank_stage1" in linear_svc_validation_df.columns
):
    linear_svc_validation_df = linear_svc_validation_df.rename(
        columns={"candidate_rank_stage1": "candidate_rank_tuning"}
    )

linear_svc_validation_view = linear_svc_validation_df[
    linear_svc_validation_view_cols
].copy()
linear_svc_best_validation = linear_svc_validation_df.sort_values(
    by=["mean_macro_f1", "mean_accuracy"],
    ascending=[False, False],
).head(1)

linear_svc_best_params = select_best_linear_svc_params(
    linear_svc_tuning_df, linear_svc_validation_df
)

print("Task 3 Validation finalists:")
display(linear_svc_validation_view)

print("Task 3 selected best Validation row:")
display(linear_svc_best_validation[linear_svc_validation_view_cols])

Task 3 Validation finalists:


,candidate_rank_tuning,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,param_clf__C,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range
0,1,0.677504,0.001772,0.761505,0.002262,0.2,150000,2,"(1, 2)"
1,2,0.675393,0.001659,0.760219,0.002083,0.2,100000,2,"(1, 2)"
2,4,0.675393,0.001659,0.760219,0.002083,0.2,100000,2,"(1, 2)"
3,3,0.673467,0.002971,0.758889,0.002344,0.2,75000,2,"(1, 2)"
4,5,0.673467,0.002971,0.758889,0.002344,0.2,75000,2,"(1, 2)"


Task 3 selected best Validation row:


,candidate_rank_tuning,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,param_clf__C,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range
0,1,0.677504,0.001772,0.761505,0.002262,0.2,150000,2,"(1, 2)"


### Test Submission Generation


In [45]:
if RUN_LINEAR_SVC_SUBMISSION_GENERATION:
    print("Task 3 best params for submission:", linear_svc_best_params)

    linear_svc_tfidf_params = {
        **TFIDF_DEFAULTS,
        "ngram_range": linear_svc_best_params.get("tfidf__ngram_range", (1, 1)),
        "min_df": linear_svc_best_params.get(
            "tfidf__min_df", TFIDF_DEFAULTS.get("min_df", 2)
        ),
        "sublinear_tf": linear_svc_best_params.get(
            "tfidf__sublinear_tf", TFIDF_DEFAULTS.get("sublinear_tf", True)
        ),
        "max_features": linear_svc_best_params.get(
            "tfidf__max_features", TFIDF_DEFAULTS.get("max_features")
        ),
    }
    linear_svc_clf_params = {
        "C": linear_svc_best_params.get("clf__C", 1.0),
        "class_weight": linear_svc_best_params.get("clf__class_weight", "balanced"),
        "max_iter": LINEAR_SVC_MAX_ITER,
        "random_state": RANDOM_STATE,
    }

    linear_svc_vectorizer = TfidfVectorizer(**linear_svc_tfidf_params)
    linear_svc_X_train = linear_svc_vectorizer.fit_transform(
        train_df["cleaned_abstract"].to_numpy(dtype=str)
    )
    linear_svc_X_test = linear_svc_vectorizer.transform(
        test_df["cleaned_abstract"].to_numpy(dtype=str)
    )
    linear_svc_y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    linear_svc_clf = LinearSVC(**linear_svc_clf_params)
    linear_svc_clf.fit(linear_svc_X_train, linear_svc_y_train)
    linear_svc_y_pred = linear_svc_clf.predict(linear_svc_X_test).astype(np.int64)

    linear_svc_submission_df = pd.DataFrame(
        {
            "id": test_df["id"].astype(np.int64),
            "label_id": linear_svc_y_pred,
        }
    )
    if linear_svc_submission_df.columns.tolist() != ["id", "label_id"]:
        raise ValueError("Task 3 submission columns must be exactly: id,label_id")
    if len(linear_svc_submission_df) != len(test_df):
        raise ValueError("Task 3 submission row count mismatch.")

    linear_svc_submission_df.to_csv(LINEAR_SVC_SUBMISSION_PATH, index=False)
    print(f"Saved Task 3 submission to: {LINEAR_SVC_SUBMISSION_PATH}")

## Complement Naive Bayes

### Configuration

In [4]:
CNB_OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "task3"
CNB_TUNING_OUTPUT_PATH = CNB_OUTPUTS_DIR / "complement_nb_halving_results.csv"
CNB_VALIDATION_OUTPUT_PATH = CNB_OUTPUTS_DIR / "complement_nb_finalists_cv5.csv"
CNB_SUBMISSION_PATH = SUBMISSIONS_DIR / "task3_complement_nb.csv"

CNB_N_JOBS = 2
CNB_SCORING = {"macro_f1": "f1_macro", "accuracy": "accuracy"}
CNB_TUNING_CV_SPLITS = 3
CNB_TUNING_FACTOR = 3
CNB_VALIDATION_CV_SPLITS = 5
CNB_VALIDATION_TOP_K = 5


# Hyperparameter search grid
CNB_TUNING_PARAM_GRID = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 5],
    "tfidf__max_features": [50000, 75000, 100000, 150000],
    "tfidf__sublinear_tf": [True, False],
    "clf__alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0],
    "clf__norm": [False, True],
}

RECOMPUTE_CNB_RESULTS = False
RUN_CNB_SUBMISSION_GENERATION = False

### Utility Functions

In [5]:
from sklearn.naive_bayes import ComplementNB


def build_cnb_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(**TFIDF_DEFAULTS)),
            ("clf", ComplementNB()),
        ]
    )


def _cnb_normalize_param_value(key: str, value):
    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if text == "":
            parsed = value
        elif text.lower() == "none":
            parsed = None
        else:
            try:
                parsed = ast.literal_eval(text)
            except ValueError, SyntaxError:
                parsed = value
    if key in {"tfidf__max_features", "tfidf__min_df"}:
        if parsed is None:
            return None
        if isinstance(parsed, (int, float, np.integer, np.floating)):
            parsed_float = float(parsed)
            if parsed_float.is_integer():
                return int(parsed_float)
    return parsed


def _cnb_extract_param_dict(row: pd.Series) -> dict:
    params = {}
    for col, value in row.items():
        if not col.startswith("param_") or pd.isna(value):
            continue
        key = col.replace("param_", "", 1)
        params[key] = _cnb_normalize_param_value(key, value)
    return params


def _cnb_top_params_from_tuning(tuning_df: pd.DataFrame, top_k: int) -> list[dict]:
    sort_col = (
        "rank_test_score"
        if "rank_test_score" in tuning_df.columns
        else "mean_test_score"
    )
    ascending = sort_col == "rank_test_score"
    top_df = tuning_df.sort_values(by=sort_col, ascending=ascending).head(top_k)
    return [_cnb_extract_param_dict(row) for _, row in top_df.iterrows()]

### Tuning

In [6]:
def run_cnb_tuning(train_df: pd.DataFrame, recompute: bool) -> pd.DataFrame:
    if not recompute:
        return pd.read_csv(CNB_TUNING_OUTPUT_PATH)

    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    cv = StratifiedKFold(
        n_splits=CNB_TUNING_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    search = HalvingGridSearchCV(
        estimator=build_cnb_pipeline(),
        param_grid=CNB_TUNING_PARAM_GRID,
        cv=cv,
        scoring="f1_macro",
        n_jobs=CNB_N_JOBS,
        factor=CNB_TUNING_FACTOR,
        error_score="raise",
        refit=True,
    )
    search.fit(X, y)

    tuning_df = (
        pd.DataFrame(search.cv_results_)
        .sort_values(by=["rank_test_score", "mean_test_score"], ascending=[True, False])
        .reset_index(drop=True)
    )
    CNB_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    tuning_df.to_csv(CNB_TUNING_OUTPUT_PATH, index=False)
    return tuning_df


cnb_tuning_df = run_cnb_tuning(train_df, recompute=RECOMPUTE_CNB_RESULTS)

sort_col = (
    "rank_test_score"
    if "rank_test_score" in cnb_tuning_df.columns
    else "mean_test_score"
)
ascending = sort_col == "rank_test_score"
cnb_best_tuning = cnb_tuning_df.sort_values(by=sort_col, ascending=ascending).head(1)

print("ComplementNB best Tuning config:")
display(
    cnb_best_tuning[
        [
            sort_col,
            "mean_test_score",
            "param_clf__alpha",
            "param_clf__norm",
            "param_tfidf__max_features",
            "param_tfidf__min_df",
            "param_tfidf__ngram_range",
            "param_tfidf__sublinear_tf",
        ]
    ]
)

ComplementNB best Tuning config:


,rank_test_score,mean_test_score,param_clf__alpha,param_clf__norm,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range,param_tfidf__sublinear_tf
0,1,0.549451,0.05,True,150000,2,"(1, 2)",False


### Validation

In [7]:
def run_cnb_validation(train_df: pd.DataFrame, tuning_df: pd.DataFrame) -> pd.DataFrame:
    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    candidates = _cnb_top_params_from_tuning(tuning_df, top_k=CNB_VALIDATION_TOP_K)
    cv = StratifiedKFold(
        n_splits=CNB_VALIDATION_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rows = []
    for idx, params in enumerate(candidates, start=1):
        pipe = build_cnb_pipeline()
        pipe.set_params(**params)
        cv_result = cross_validate(
            estimator=pipe,
            X=X,
            y=y,
            cv=cv,
            scoring=CNB_SCORING,
            n_jobs=CNB_N_JOBS,
            return_train_score=False,
            error_score="raise",
        )
        row = {
            "candidate_rank_tuning": idx,
            "mean_macro_f1": float(np.mean(cv_result["test_macro_f1"])),
            "std_macro_f1": float(np.std(cv_result["test_macro_f1"])),
            "mean_accuracy": float(np.mean(cv_result["test_accuracy"])),
            "std_accuracy": float(np.std(cv_result["test_accuracy"])),
            "mean_fit_time_sec": float(np.mean(cv_result["fit_time"])),
            "mean_score_time_sec": float(np.mean(cv_result["score_time"])),
        }
        for key, value in params.items():
            row[f"param_{key}"] = value
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(by=["mean_macro_f1", "mean_accuracy"], ascending=[False, False])
        .reset_index(drop=True)
    )


def select_best_cnb_params(
    tuning_df: pd.DataFrame,
    validation_df: pd.DataFrame,
) -> dict:
    if not validation_df.empty:
        best = validation_df.sort_values(
            by=["mean_macro_f1", "mean_accuracy"],
            ascending=[False, False],
        ).iloc[0]
        return _cnb_extract_param_dict(best)
    sort_col = (
        "rank_test_score"
        if "rank_test_score" in tuning_df.columns
        else "mean_test_score"
    )
    ascending = sort_col == "rank_test_score"
    best = tuning_df.sort_values(by=sort_col, ascending=ascending).iloc[0]
    return _cnb_extract_param_dict(best)


if RECOMPUTE_CNB_RESULTS:
    cnb_validation_df = run_cnb_validation(train_df, cnb_tuning_df)
    cnb_validation_df.to_csv(CNB_VALIDATION_OUTPUT_PATH, index=False)
else:
    cnb_validation_df = pd.read_csv(CNB_VALIDATION_OUTPUT_PATH)

if (
    "candidate_rank_tuning" not in cnb_validation_df.columns
    and "candidate_rank_stage1" in cnb_validation_df.columns
):
    cnb_validation_df = cnb_validation_df.rename(
        columns={"candidate_rank_stage1": "candidate_rank_tuning"}
    )

cnb_validation_view_cols = [
    "candidate_rank_tuning",
    "mean_macro_f1",
    "std_macro_f1",
    "mean_accuracy",
    "std_accuracy",
    "param_clf__alpha",
    "param_clf__norm",
    "param_tfidf__max_features",
    "param_tfidf__ngram_range",
]
cnb_validation_view = cnb_validation_df[
    [c for c in cnb_validation_view_cols if c in cnb_validation_df.columns]
].copy()
cnb_best_validation = cnb_validation_df.sort_values(
    by=["mean_macro_f1", "mean_accuracy"], ascending=[False, False]
).head(1)
cnb_best_params = select_best_cnb_params(cnb_tuning_df, cnb_validation_df)

print("ComplementNB Validation finalists:")
display(cnb_validation_view)

print("ComplementNB selected best Validation row:")
display(
    cnb_best_validation[
        [c for c in cnb_validation_view_cols if c in cnb_best_validation.columns]
    ]
)

ComplementNB Validation finalists:


,candidate_rank_tuning,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,param_clf__alpha,param_clf__norm,param_tfidf__max_features,param_tfidf__ngram_range
0,1,0.565427,0.001888,0.722887,0.001572,0.05,True,150000,"(1, 2)"
1,2,0.559376,0.002438,0.720113,0.001408,0.01,True,100000,"(1, 2)"
2,3,0.549277,0.001702,0.715118,0.001249,0.01,True,75000,"(1, 2)"
3,4,0.547886,0.000933,0.714536,0.001623,0.05,True,75000,"(1, 2)"
4,5,0.545765,0.001395,0.715413,0.001634,0.05,True,75000,"(1, 2)"


ComplementNB selected best Validation row:


,candidate_rank_tuning,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,param_clf__alpha,param_clf__norm,param_tfidf__max_features,param_tfidf__ngram_range
0,1,0.565427,0.001888,0.722887,0.001572,0.05,True,150000,"(1, 2)"


### Test Submission Generation

In [8]:
if RUN_CNB_SUBMISSION_GENERATION:
    print("ComplementNB best params for submission:", cnb_best_params)

    cnb_tfidf_params = {
        **TFIDF_DEFAULTS,
        "ngram_range": cnb_best_params.get("tfidf__ngram_range", (1, 1)),
        "min_df": cnb_best_params.get("tfidf__min_df", TFIDF_DEFAULTS.get("min_df", 2)),
        "sublinear_tf": cnb_best_params.get(
            "tfidf__sublinear_tf", TFIDF_DEFAULTS.get("sublinear_tf", True)
        ),
        "max_features": cnb_best_params.get(
            "tfidf__max_features", TFIDF_DEFAULTS.get("max_features")
        ),
    }
    cnb_clf_params = {
        "alpha": cnb_best_params.get("clf__alpha", 0.05),
        "norm": cnb_best_params.get("clf__norm", True),
    }

    cnb_vectorizer = TfidfVectorizer(**cnb_tfidf_params)
    cnb_X_train = cnb_vectorizer.fit_transform(
        train_df["cleaned_abstract"].to_numpy(dtype=str)
    )
    cnb_X_test = cnb_vectorizer.transform(
        test_df["cleaned_abstract"].to_numpy(dtype=str)
    )
    cnb_y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    cnb_clf = ComplementNB(**cnb_clf_params)
    cnb_clf.fit(cnb_X_train, cnb_y_train)
    cnb_y_pred = cnb_clf.predict(cnb_X_test).astype(np.int64)

    cnb_submission_df = pd.DataFrame(
        {
            "id": test_df["id"].astype(np.int64),
            "label_id": cnb_y_pred,
        }
    )
    if cnb_submission_df.columns.tolist() != ["id", "label_id"]:
        raise ValueError("ComplementNB submission columns must be exactly: id,label_id")
    if len(cnb_submission_df) != len(test_df):
        raise ValueError("ComplementNB submission row count mismatch.")

    cnb_submission_df.to_csv(CNB_SUBMISSION_PATH, index=False)
    print(f"Saved ComplementNB submission to: {CNB_SUBMISSION_PATH}")

## Random Forest

### Configuration

In [6]:
from sklearn.ensemble import RandomForestClassifier

RANDOM_FOREST_OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "task3"
RANDOM_FOREST_TUNING_OUTPUT_PATH = (
    RANDOM_FOREST_OUTPUTS_DIR / "random_forest_halving_results.csv"
)
RANDOM_FOREST_VALIDATION_OUTPUT_PATH = (
    RANDOM_FOREST_OUTPUTS_DIR / "random_forest_finalists_cv5.csv"
)
RANDOM_FOREST_SUBMISSION_PATH = SUBMISSIONS_DIR / "task3_random_forest.csv"

RANDOM_FOREST_N_JOBS = 2
RANDOM_FOREST_SCORING = {"macro_f1": "f1_macro", "accuracy": "accuracy"}
RANDOM_FOREST_TUNING_CV_SPLITS = 3
RANDOM_FOREST_TUNING_FACTOR = 3
RANDOM_FOREST_VALIDATION_CV_SPLITS = 5
RANDOM_FOREST_VALIDATION_TOP_K = 5
RANDOM_FOREST_TUNING_PARAM_GRID = {
    "tfidf__ngram_range": [(1, 2)],
    "tfidf__min_df": [2, 5],
    "tfidf__max_features": [75000, 100000, 150000, 200000],
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [15, 20, 25],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2],
}

RECOMPUTE_RANDOM_FOREST_RESULTS = False
RUN_RANDOM_FOREST_SUBMISSION_GENERATION = False

### Utility Function

In [7]:
def build_random_forest_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(**TFIDF_DEFAULTS)),
            (
                "clf",
                RandomForestClassifier(
                    random_state=RANDOM_STATE,
                    class_weight="balanced",
                    n_jobs=RANDOM_FOREST_N_JOBS,
                ),
            ),
        ]
    )


def _random_forest_normalize_param_value(key: str, value):
    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if text == "":
            parsed = value
        elif text.lower() == "none":
            parsed = None
        else:
            try:
                parsed = ast.literal_eval(text)
            except ValueError, SyntaxError:
                parsed = value

    if key in {"tfidf__max_features", "tfidf__min_df", "clf__n_estimators", "clf__max_depth", "clf__min_samples_split", "clf__min_samples_leaf"}:
        if parsed is None:
            return None
        if isinstance(parsed, (int, float, np.integer, np.floating)):
            parsed_float = float(parsed)
            if parsed_float.is_integer():
                return int(parsed_float)
    return parsed


def _random_forest_extract_param_dict(row: pd.Series) -> dict:
    params = {}
    for col, value in row.items():
        if not col.startswith("param_") or pd.isna(value):
            continue
        key = col.replace("param_", "", 1)
        params[key] = _random_forest_normalize_param_value(key, value)
    return params


def _random_forest_top_params_from_tuning(
    tuning_df: pd.DataFrame, top_k: int
) -> list[dict]:
    sort_col = (
        "rank_test_score"
        if "rank_test_score" in tuning_df.columns
        else "mean_test_score"
    )
    ascending = sort_col == "rank_test_score"
    top_df = tuning_df.sort_values(by=sort_col, ascending=ascending).head(top_k)
    return [_random_forest_extract_param_dict(row) for _, row in top_df.iterrows()]

### Tuning

In [8]:
def run_random_forest_tuning(train_df: pd.DataFrame, recompute: bool) -> pd.DataFrame:
    if not recompute:
        return pd.read_csv(RANDOM_FOREST_TUNING_OUTPUT_PATH)

    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    cv = StratifiedKFold(
        n_splits=RANDOM_FOREST_TUNING_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    search = HalvingGridSearchCV(
        estimator=build_random_forest_pipeline(),
        param_grid=RANDOM_FOREST_TUNING_PARAM_GRID,
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,  # Set to 1 because RandomForest already uses n_jobs internally
        factor=RANDOM_FOREST_TUNING_FACTOR,
        error_score="raise",
        refit=True,
    )
    search.fit(X, y)

    tuning_df = (
        pd.DataFrame(search.cv_results_)
        .sort_values(by=["rank_test_score", "mean_test_score"], ascending=[True, False])
        .reset_index(drop=True)
    )
    RANDOM_FOREST_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    tuning_df.to_csv(RANDOM_FOREST_TUNING_OUTPUT_PATH, index=False)
    return tuning_df


random_forest_tuning_df = run_random_forest_tuning(
    train_df,
    recompute=RECOMPUTE_RANDOM_FOREST_RESULTS,
)

sort_col = (
    "rank_test_score"
    if "rank_test_score" in random_forest_tuning_df.columns
    else "mean_test_score"
)
ascending = sort_col == "rank_test_score"
random_forest_best_tuning = random_forest_tuning_df.sort_values(
    by=sort_col, ascending=ascending
).head(1)

print("Task 3 Random Forest best Tuning config:")
display(
    random_forest_best_tuning[
        [
            sort_col,
            "mean_test_score",
            "param_clf__n_estimators",
            "param_clf__max_depth",
            "param_clf__min_samples_split",
            "param_clf__min_samples_leaf",
            "param_tfidf__max_features",
            "param_tfidf__min_df",
            "param_tfidf__ngram_range",
        ]
    ]
)

Task 3 Random Forest best Tuning config:


,rank_test_score,mean_test_score,param_clf__n_estimators,param_clf__max_depth,param_clf__min_samples_split,param_clf__min_samples_leaf,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range
0,1,0.477051,300,25,5,1,100000,2,"(1, 2)"


### Validation

In [9]:
def run_random_forest_validation(
    train_df: pd.DataFrame, tuning_df: pd.DataFrame
) -> pd.DataFrame:
    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    candidates = _random_forest_top_params_from_tuning(
        tuning_df, top_k=RANDOM_FOREST_VALIDATION_TOP_K
    )
    cv = StratifiedKFold(
        n_splits=RANDOM_FOREST_VALIDATION_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rows = []
    for idx, params in enumerate(candidates, start=1):
        pipe = build_random_forest_pipeline()
        pipe.set_params(**params)
        cv_result = cross_validate(
            estimator=pipe,
            X=X,
            y=y,
            cv=cv,
            scoring=RANDOM_FOREST_SCORING,
            n_jobs=1,  # Set to 1 because RandomForest already uses n_jobs internally
            return_train_score=False,
            error_score="raise",
        )

        row = {
            "candidate_rank_tuning": idx,
            "mean_macro_f1": float(np.mean(cv_result["test_macro_f1"])),
            "std_macro_f1": float(np.std(cv_result["test_macro_f1"])),
            "mean_accuracy": float(np.mean(cv_result["test_accuracy"])),
            "std_accuracy": float(np.std(cv_result["test_accuracy"])),
            "mean_fit_time_sec": float(np.mean(cv_result["fit_time"])),
            "mean_score_time_sec": float(np.mean(cv_result["score_time"])),
        }
        for key, value in params.items():
            row[f"param_{key}"] = value
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(by=["mean_macro_f1", "mean_accuracy"], ascending=[False, False])
        .reset_index(drop=True)
    )


def select_best_random_forest_params(
    tuning_df: pd.DataFrame,
    validation_df: pd.DataFrame,
) -> dict:
    if not validation_df.empty:
        best = validation_df.sort_values(
            by=["mean_macro_f1", "mean_accuracy"],
            ascending=[False, False],
        ).iloc[0]
        return _random_forest_extract_param_dict(best)

    sort_col = (
        "rank_test_score"
        if "rank_test_score" in tuning_df.columns
        else "mean_test_score"
    )
    ascending = sort_col == "rank_test_score"
    best = tuning_df.sort_values(by=sort_col, ascending=ascending).iloc[0]
    return _random_forest_extract_param_dict(best)


if RECOMPUTE_RANDOM_FOREST_RESULTS:
    random_forest_validation_df = run_random_forest_validation(train_df, random_forest_tuning_df)
    random_forest_validation_df.to_csv(RANDOM_FOREST_VALIDATION_OUTPUT_PATH, index=False)
else:
    random_forest_validation_df = pd.read_csv(RANDOM_FOREST_VALIDATION_OUTPUT_PATH)

random_forest_validation_view_cols = [
    "candidate_rank_tuning",
    "mean_macro_f1",
    "std_macro_f1",
    "mean_accuracy",
    "std_accuracy",
    "param_clf__n_estimators",
    "param_clf__max_depth",
    "param_clf__min_samples_split",
    "param_clf__min_samples_leaf",
    "param_tfidf__max_features",
    "param_tfidf__min_df",
    "param_tfidf__ngram_range",
]
if (
    "candidate_rank_tuning" not in random_forest_validation_df.columns
    and "candidate_rank_stage1" in random_forest_validation_df.columns
):
    random_forest_validation_df = random_forest_validation_df.rename(
        columns={"candidate_rank_stage1": "candidate_rank_tuning"}
    )

random_forest_validation_view = random_forest_validation_df[
    random_forest_validation_view_cols
].copy()
random_forest_best_validation = random_forest_validation_df.sort_values(
    by=["mean_macro_f1", "mean_accuracy"],
    ascending=[False, False],
).head(1)

random_forest_best_params = select_best_random_forest_params(
    random_forest_tuning_df, random_forest_validation_df
)

print("Task 3 Random Forest Validation finalists:")
display(random_forest_validation_view)

print("Task 3 Random Forest selected best Validation row:")
display(random_forest_best_validation[random_forest_validation_view_cols])

Task 3 Random Forest Validation finalists:


,candidate_rank_tuning,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,param_clf__n_estimators,param_clf__max_depth,param_clf__min_samples_split,param_clf__min_samples_leaf,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range
0,2,0.497524,0.005378,0.588649,0.002683,300,25,5,1,200000,2,"(1, 2)"
1,3,0.497460,0.004232,0.589001,0.003140,300,25,5,1,150000,2,"(1, 2)"
2,4,0.496032,0.002597,0.587693,0.001229,300,25,5,1,150000,5,"(1, 2)"
3,5,0.495939,0.003362,0.588814,0.002154,300,25,5,1,100000,5,"(1, 2)"
4,1,0.494259,0.005272,0.588103,0.004263,300,25,5,1,100000,2,"(1, 2)"


Task 3 Random Forest selected best Validation row:


,candidate_rank_tuning,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,param_clf__n_estimators,param_clf__max_depth,param_clf__min_samples_split,param_clf__min_samples_leaf,param_tfidf__max_features,param_tfidf__min_df,param_tfidf__ngram_range
0,2,0.497524,0.005378,0.588649,0.002683,300,25,5,1,200000,2,"(1, 2)"


### Test Submission Generation

In [10]:
if RUN_RANDOM_FOREST_SUBMISSION_GENERATION:
    print("Task 3 Random Forest best params for submission:", random_forest_best_params)

    random_forest_tfidf_params = {
        **TFIDF_DEFAULTS,
        "ngram_range": random_forest_best_params.get("tfidf__ngram_range", (1, 1)),
        "min_df": random_forest_best_params.get(
            "tfidf__min_df", TFIDF_DEFAULTS.get("min_df", 2)
        ),
        "sublinear_tf": random_forest_best_params.get(
            "tfidf__sublinear_tf", TFIDF_DEFAULTS.get("sublinear_tf", True)
        ),
        "max_features": random_forest_best_params.get(
            "tfidf__max_features", TFIDF_DEFAULTS.get("max_features")
        ),
    }
    random_forest_clf_params = {
        "n_estimators": random_forest_best_params.get("clf__n_estimators", 100),
        "max_depth": random_forest_best_params.get("clf__max_depth", None),
        "min_samples_split": random_forest_best_params.get("clf__min_samples_split", 2),
        "min_samples_leaf": random_forest_best_params.get("clf__min_samples_leaf", 1),
        "class_weight": random_forest_best_params.get("clf__class_weight", "balanced"),
        "random_state": RANDOM_STATE,
        "n_jobs": RANDOM_FOREST_N_JOBS,
    }

    random_forest_vectorizer = TfidfVectorizer(**random_forest_tfidf_params)
    random_forest_X_train = random_forest_vectorizer.fit_transform(
        train_df["cleaned_abstract"].to_numpy(dtype=str)
    )
    random_forest_X_test = random_forest_vectorizer.transform(
        test_df["cleaned_abstract"].to_numpy(dtype=str)
    )
    random_forest_y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    random_forest_clf = RandomForestClassifier(**random_forest_clf_params)
    random_forest_clf.fit(random_forest_X_train, random_forest_y_train)
    random_forest_y_pred = random_forest_clf.predict(random_forest_X_test).astype(np.int64)

    random_forest_submission_df = pd.DataFrame(
        {
            "id": test_df["id"].astype(np.int64),
            "label_id": random_forest_y_pred,
        }
    )
    if random_forest_submission_df.columns.tolist() != ["id", "label_id"]:
        raise ValueError("Task 3 Random Forest submission columns must be exactly: id,label_id")
    if len(random_forest_submission_df) != len(test_df):
        raise ValueError("Task 3 Random Forest submission row count mismatch.")

    random_forest_submission_df.to_csv(RANDOM_FOREST_SUBMISSION_PATH, index=False)
    print(f"Saved Task 3 Random Forest submission to: {RANDOM_FOREST_SUBMISSION_PATH}")

## XGBoost

### Configuration

In [7]:
import optuna
import xgboost as xgb

# XGBoost/Optuna File Paths
XGB_OPTUNA_DIR = PROJECT_ROOT / "outputs" / "task3"
XGB_OPTUNA_RESULTS_PATH = XGB_OPTUNA_DIR / "xgboost_optuna_best.csv"
XGB_OPTUNA_SUBMISSION_PATH = SUBMISSIONS_DIR / "task3_xgboost_optuna_final.csv"

# Study Settings
N_TRIALS = 20
SAMPLE_FRAC = 0.5  # Fraction of data to use for search to save time

# Run Controls
RECOMPUTE_OPTUNA_SEARCH = False
RUN_OPTUNA_SUBMISSION_GENERATION = False

c:\Users\esthe\OneDrive\Documents\term 6\ml\ml proj\ml-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Optimization (Optuna Study)

In [8]:
def objective(trial, X, y):
    """The function Optuna tries to maximize."""
    # Define the search space
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1000),
        "max_depth": trial.suggest_int("max_depth", 6, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.7),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "objective": "multi:softmax",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": 1, # Leave fold-level parallelism to sklearn
    }

    max_features = trial.suggest_categorical("max_features", [80000, 120000, 150000])

    # Vectorization
    tfidf_config = {**TFIDF_DEFAULTS, "max_features": max_features, "sublinear_tf": True}
    tfidf = TfidfVectorizer(**tfidf_config)
    X_transformed = tfidf.fit_transform(X)

    # 3-Fold Cross-Validation to evaluate the trial
    clf = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(clf, X_transformed, y, cv=cv, scoring="f1_macro", n_jobs=-1)

    return scores.mean()

def run_optuna_search(train_df):
    """Runs the optimization study on a sampled portion of the data."""
    print(f"Sampling {SAMPLE_FRAC:.0%} of the training set for Optuna...")
    sampled_df = train_df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)

    X = sampled_df["cleaned_abstract"].values.astype(str)
    # XGBoost requires consecutive labels starting at 0
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(sampled_df["label_id"])

    print(f"Starting Optuna Study ({N_TRIALS} trials)...")
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, X, y), n_trials=N_TRIALS)

    print("\nBest Trial Results:")
    print(f"  Macro F1: {study.best_value:.4f}")

    XGB_OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
    best_params_df = pd.DataFrame([study.best_params])
    best_params_df.to_csv(XGB_OPTUNA_RESULTS_PATH, index=False)
    return best_params_df

# Execution
if RECOMPUTE_OPTUNA_SEARCH:
    xgb_best_params_df = run_optuna_search(train_df)
else:
    # If not recomputing, load existing winning CSV
    xgb_best_params_df = pd.read_csv(XGB_OPTUNA_RESULTS_PATH)

print("Task 3 Optuna Winning Parameters:")
display(xgb_best_params_df)

Task 3 Optuna Winning Parameters:


,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,max_features
0,530,9,0.066719,0.638786,0.515319,2,80000


### Test Submission Generation

In [9]:
if RUN_OPTUNA_SUBMISSION_GENERATION:
    best_params = xgb_best_params_df.iloc[0].to_dict()

    # 1. Prepare 100% dataset features
    train_texts = train_df["cleaned_abstract"].values.astype(str)
    test_texts = test_df["cleaned_abstract"].values.astype(str)
    test_ids = test_df["id"].values.astype(np.int64)

    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_train = le.fit_transform(train_df["label_id"])

    # 2. Vectorization with winning max_features
    tfidf_config = TFIDF_DEFAULTS.copy()
    tfidf_config.update({
        "max_features": int(best_params["max_features"]),
        "sublinear_tf": True
    })

    print(f"Vectorizing full data with max_features={tfidf_config['max_features']}...")
    vectorizer = TfidfVectorizer(**tfidf_config)
    X_train_full = vectorizer.fit_transform(train_texts)
    X_test_full = vectorizer.transform(test_texts)

    # 3. Final Model Training
    print(f"Training final model with {int(best_params['n_estimators'])} trees...")
    clf_final = xgb.XGBClassifier(
        n_estimators=int(best_params["n_estimators"]),
        max_depth=int(best_params["max_depth"]),
        learning_rate=float(best_params["learning_rate"]),
        subsample=float(best_params.get("subsample", 0.8)),
        colsample_bytree=float(best_params.get("colsample_bytree", 0.5)),
        min_child_weight=int(best_params.get("min_child_weight", 2)),
        objective="multi:softmax",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    clf_final.fit(X_train_full, y_train_full)

    # 4. Generate and Save Submission
    y_pred = clf_final.predict(X_test_full)
    y_pred_orig = le.inverse_transform(y_pred)

    submission_df = pd.DataFrame({
        "id": test_ids,
        "label_id": y_pred_orig.astype(np.int64),
    })

    submission_df.to_csv(XGB_OPTUNA_SUBMISSION_PATH, index=False)
    print(f"SUCCESS! Submission saved to: {XGB_OPTUNA_SUBMISSION_PATH}")